### smooth gate vs heaviside visualisation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

k1 = 1.0
k2 = 0.22
k3 = 0.12
T_peak = 5
T_max = 20.0
sharpness_list = [5.0, 10.0, 20.0]  # σ slopes for smooth gate

t = np.linspace(0, T_max, 600)

# ---- helper functions ----
def expm1_clip(x, max_in=85.0):
    return np.expm1(np.clip(x, None, max_in))


def smooth_kernel(t, T_peak, k1, k2, k3, sharpness=10.0):
    # w(t) ~ 0 before T_peak, ~1 after; smooth transition
    w = 1.0 / (1.0 + np.exp(-sharpness * (t - T_peak)))
    rise_full = expm1_clip(k2 * t)
    peak_val  = expm1_clip(k2 * T_peak)
    fall_part = np.exp(np.clip(-2.0 * k3 * (t - T_peak), None, 85.0))
    return k1 * ((1.0 - w) * rise_full + w * peak_val * fall_part), w


# ---- 1) Gate visualisation (w vs Heaviside) ----
plt.figure(figsize=(5, 2.5), dpi=300)
for s in sharpness_list:
    _, w = smooth_kernel(t, T_peak, k1, k2, k3, sharpness=s)
    plt.plot(t, w, label=f"smooth σ, s={s}")
# Heaviside gate used implicitly by the piecewise (0 before, 1 after)
H = (t > T_peak).astype(float)
plt.plot(t, H, linestyle="--", label="Heaviside (piecewise)")
plt.axvline(T_peak, linestyle=":", alpha=0.8)
plt.xlabel("t")
plt.ylabel("gate weight w(t)")
plt.title("Smooth gate vs. hard Heaviside")
plt.legend()
plt.tight_layout()
plt.show()

### shedding curve (theoretical) visualization

In [ ]:
import jax.nn as jnn
import jax.numpy as jnp
import jax

In [ ]:
T_peak = 2.9                 # days
T_max = 15                 # days
k1 = 0.1             
k2 = 0.61     
k3 = 0.61      
curve_colour = "goldenrod"  

log_k1 = np.log(k1)
k2_shift = (k2-0.6)/(2.5-0.6)
logit_k2 = jax.numpy.log(k2_shift / (1 - k2_shift))
k3_shift = (k3-0.15)/(2.0-0.15)
logit_k3 = jax.numpy.log(k3_shift / (1 - k3_shift))

T_peak_shift = (T_peak - 1)/(5 - 1)
logit_T_peak = jax.numpy.log(T_peak_shift / (1 - T_peak_shift))

In [ ]:

def _safe_exp(x):
    """Exponentiate with clipping to avoid overflow/underflow."""
    x = jnp.asarray(x)
    finfo = jnp.finfo(x.dtype)
    # Clip in *natural* log space
    x = jnp.clip(x, jnp.log(finfo.tiny), jnp.log(finfo.max))
    return jnp.exp(x)

def shedding_curve(t, logit_T_peak, log_k1, logit_k2, logit_k3):
    t = jnp.asarray(t)
    log10 = jnp.log(jnp.array(10.0, dtype=t.dtype))

    k1 = jnp.exp(log_k1)
    k2 = jnn.sigmoid(logit_k2) * (2.5 - 0.6) + 0.6
    k3 = jnn.sigmoid(logit_k3) * (2.0 - 0.15) + 0.15
    T_peak = jnn.sigmoid(logit_T_peak) * (5.0 - 1.0) + 1.0

    # Arguments in natural-log space for the 10**(.) terms
    inc_arg  = log10 * (k2 * t)
    peak_arg = log10 * (k2 * T_peak)
    dec_arg  = log10 * (-k3 * (t - T_peak))

    # These are now guaranteed finite (no inf) thanks to _safe_exp
    increase_phase = _safe_exp(inc_arg) - 1.0          # ≈ 10**(k2*t) - 1
    peak_value     = _safe_exp(peak_arg) - 1.0    # ≈ k1*10**(k2*T_peak) - 1
    decrease       = _safe_exp(dec_arg)                # ≈ 10**(-k3*(t - T_peak))

    w = jnn.sigmoid(10.0 * (t - T_peak))
    return k1 * ((1.0 - w) * increase_phase + w * peak_value * decrease)


In [ ]:
# ---- Generate data ----
t = np.linspace(0, T_max+2, 1000)
y = shedding_curve(t, logit_T_peak, log_k1, logit_k2, logit_k3)

# ---- Plot ----
plt.figure(figsize=(4, 2))  # single plot as requested
plt.plot(t, y, linewidth=2.5, color=curve_colour, label="Shedding kernel")
plt.axvline(T_peak, linestyle="--", linewidth=1.5, label=r"$T_{peak}$", c="grey")
plt.axvline(T_max, linestyle=":", linewidth=1.5, label=r"$T_{max}$", c="black")
plt.xlabel("Time since being infected [days]")
plt.ylabel("Relative shedding")

plt.grid(True, alpha=0.3)
plt.legend(frameon=True)
plt.tight_layout()
plt.savefig("shedding_curve.png", dpi=300, bbox_inches="tight")
